# GIẢI THUẬT 3: BAYES NGÂY THƠ (NAÏVE BAYES)

## 1. Ôn tập lý thuyết

**1) Giải thuật Naive Bayes hoạt động như thế nào? Hãy giải thích định lý Bayes và giả định "ngây thơ" trong thuật toán này?**

- **Định lý Bayes** (xác suất hậu nghiệm của lớp $y$ khi quan sát đặc trưng $\mathbf{x}=(x_1,\dots,x_d)$):

$$
P(y \mid \mathbf{x}) \propto P(y)\, P(\mathbf{x} \mid y)
$$

- **Giả định "ngây thơ"**: các đặc trưng **độc lập có điều kiện** theo lớp

$$
P(\mathbf{x} \mid y) = \prod_{j=1}^{d} P(x_j \mid y)
$$

- **Suy diễn dự đoán** (tính ở **log-space** để ổn định số học):

$$
\hat{y} = \operatorname*{arg\,max}_{y} \left[ \log P(y) + \sum_{j=1}^{d} \log P(x_j \mid y) \right]
$$

- $P(y)$ (prior) ước lượng theo tần suất lớp trong train.  
- $P(x_j \mid y)$ ước lượng tùy biến thể (Gaussian / Multinomial / Bernoulli).

**2) Các loại mô hình Naive Bayes (Gaussian, Multinomial, Bernoulli) khác nhau ra sao? Khi nào nên sử dụng từng loại?**

**Gaussian Naive Bayes (GNB)**  
- Cho **đặc trưng liên tục**; giả định mỗi feature|class tuân **phân phối chuẩn**.  
- Ví dụ: sensor/biomedical, dữ liệu số liên tục (đã xử lý thiếu/outliers hợp lý).

**Multinomial Naive Bayes (MNB)**  
- Cho **đặc trưng đếm không âm** (counts) như **Bag-of-Words/TF-IDF**.  
- Ứng dụng chính: **text classification**, log đếm sự kiện.

**Bernoulli Naive Bayes (BNB)**  
- Cho **đặc trưng nhị phân** (0/1): có/không.  
- Hữu ích khi vector đặc trưng là **nhị phân** (ví dụ: xuất hiện từ khóa).

> Ngoài ra: **CategoricalNB** (cho feature rời rạc dạng integer), **ComplementNB** (hữu ích với dữ liệu text lệch lớp).
**3) Tại sao Naive Bayes được gọi là "ngây thơ"? Giả định về tính độc lập của các đặc trưng ảnh hưởng như thế nào đến hiệu suất của mô hình?**
- "Ngây thơ" vì giả định **độc lập có điều kiện** của các đặc trưng theo lớp — thực tế thường **không hoàn toàn đúng**.  
- Dù vậy, Naive Bayes vẫn hoạt động **rất tốt** trong nhiều bối cảnh (đặc biệt text) vì số chiều lớn và tính thưa khiến các phụ thuộc yếu đi.  
- Nếu vi phạm độc lập **rất mạnh**, ước lượng **xác suất** có thể lệch; tuy nhiên **xếp hạng** lớp vẫn ổn → **độ chính xác phân loại** thường khá.
**4) Ưu điểm và hạn chế của Naive Bayes so với các thuật toán phân loại khác như SVM hoặc Random Forest là gì?**
**Ưu điểm**  
- **Rất nhanh & nhẹ** (train/predict), dễ triển khai, ít tham số.  
- **Cực hợp** dữ liệu **lớn/thưa** (đặc biệt văn bản).  
- Trả ra **xác suất** (dù đôi khi chưa được hiệu chỉnh tốt).

**Hạn chế**  
- Giả định độc lập mạnh → **xác suất** có thể **không chuẩn**.  
- Với dữ liệu tabular phi tuyến/phụ thuộc phức tạp, **SVM/RF/GBM** thường **tốt hơn**.  
- GNB kém khi feature liên tục **không-Gaussian**.

**5) Viết đoạn code mẫu bằng Python (sử dụng Scikit-learn) để xây dựng một mô hình Naive Bayes (ví dụ: Gaussian Naive Bayes) không? Hãy mô tả các bước thực hiện**
- Quy trình: tách dữ liệu → chia train/test (stratify) → pipeline (impute/scale nếu cần) → fit/predict → đánh giá.


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB

# 1) Dữ liệu liên tục synthetic
X, y = make_classification(n_samples=1200, n_features=8, n_informative=5, 
                           n_redundant=1, n_clusters_per_class=2, random_state=42)
df = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
df["target"] = y

# 2) Train/Test split (stratify để giữ tỉ lệ lớp)
X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3) Pipeline: impute (nếu có thiếu) + GaussianNB
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", GaussianNB(var_smoothing=1e-9))
])

# 4) Train & Evaluate
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("== Classification Report (GNB) ==")
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

== Classification Report (GNB) ==
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       121
           1       0.82      0.83      0.83       119

    accuracy                           0.83       240
   macro avg       0.83      0.83      0.83       240
weighted avg       0.83      0.83      0.83       240

Confusion matrix:
 [[100  21]
 [ 20  99]]


**6) Làm thế nào để xử lý dữ liệu phân loại (categorical data) trước khi áp dụng Multinomial Naive Bayes trong Python? **MultinomialNB** kỳ vọng **đặc trưng đếm không âm**. Cách phổ biến: **One-Hot Encoding** các cột phân loại → vector 0/1 **không âm** (coi như đếm). Có thể dùng **CategoricalNB** nếu toàn bộ đặc trưng rời rạc (mã hóa integer).**


In [2]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# Dataset toy (categorical + một cột số không âm)
df_cat = pd.DataFrame({
    "color": ["red","red","blue","green","blue","green","red","green","blue","red"],
    "size":  ["S","M","M","L","S","L","M","S","L","M"],
    "count": [1,2,1,3,2,1,2,1,3,2],   # không âm
    "target":[0,1,0,1,0,1,1,0,1,1]
})

X = df_cat.drop(columns=["target"])
y = df_cat["target"]

cat_cols = X.select_dtypes(include=["object","category"]).columns
num_cols = X.select_dtypes(include=["number"]).columns

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)  # đã không âm
])

pipe = Pipeline([
    ("prep", preprocess),
    ("clf", MultinomialNB(alpha=1.0))
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
pipe.fit(Xtr, ytr)
yp = pipe.predict(Xte)
print("== Classification Report (MultinomialNB + One-Hot) ==")
print(classification_report(yte, yp))


== Classification Report (MultinomialNB + One-Hot) ==
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



**7) Naive Bayes thường được sử dụng trong phân loại văn bản (text classification). Bạn có thể giải thích cách triển khai Naive Bayes cho bài toán này không?**
Quy trình: tiền xử lý → **TfidfVectorizer** (hoặc CountVectorizer) → **MultinomialNB** → đánh giá.

In [3]:

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# Tập văn bản nhỏ (demo)
docs = [
    "khuyến mãi cực sốc giảm giá 50% mua ngay",
    "nhận quà tặng miễn phí khi đăng ký tài khoản",
    "lịch họp dự án chiều nay lúc 3 giờ",
    "báo cáo tiến độ tuần này đã cập nhật",
    "vay tiền lãi suất thấp giải ngân trong ngày",
    "mời tham dự hội thảo khoa học vào thứ sáu",
    "giảm giá sốc duy nhất hôm nay click vào link",
    "kế hoạch đào tạo nội bộ tháng tới",
]
labels = ["spam","spam","ham","ham","spam","ham","spam","ham"]

Xtr, Xte, ytr, yte = train_test_split(docs, labels, test_size=0.25, stratify=labels, random_state=42)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1, max_df=0.95)),
    ("clf", MultinomialNB(alpha=0.5))
])

pipe.fit(Xtr, ytr)
yp = pipe.predict(Xte)

print("== Classification Report (Text: TF-IDF + MultinomialNB) ==")
print(classification_report(yte, yp))


== Classification Report (Text: TF-IDF + MultinomialNB) ==
              precision    recall  f1-score   support

         ham       0.50      1.00      0.67         1
        spam       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



c:\Users\luong\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\luong\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\luong\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 2. Bài làm mẫu

### 2.1 Bài toán 1: Xây dựng mô hình dữ liệu bằng giải thuật Bayes ngây thơ

#### 2.1.1 Nhiệm vụ 1: Phân loại sử dụng Naïve Bays

##### 1. Import thư viện và nạp dữ liệu vào notebook

In [7]:
#Importing the Necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
data = pd.read_csv('spam.csv', encoding='latin-1')
#display the first 5 rows
data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


##### 2. Xử lý dữ liệu trước khi xây dựng mô hình từ dữ liệu

In [8]:
# Drop the columns with NaN values
data = data.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'],axis=1)
# Rename columns for clarity:
data.columns = ['label', 'text']
# Separate features (X) and target labels (y)
X = data.drop('label', axis=1)
y = data['label']
# Split the data into training and testing sets (80% training, 20%testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)

**Nhận xét:**

##### 3. Xây dựng vector hóa nội dung HAM | SPAM của tập train và tập test

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()

# Fit and transform the training data (X_train)
X_train_vectorized = vectorizer.fit_transform(X_train['text'])
# Transform the test data (X_test)
X_test_vectorized = vectorizer.transform(X_test['text'])

**Nhận xét:**

##### 4. Xây dựng mô hình Naïve Bayes

In [10]:
from sklearn.naive_bayes import MultinomialNB
classifier = MultinomialNB()
classifier.fit(X_train_vectorized, y_train)

MultinomialNB()

**Nhận xét:**

##### 5. Đánh giá hiệu quả của mô hình

In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
# Make predictions on the test data
y_pred = classifier.predict(X_test_vectorized)
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(classification_rep)

Accuracy: 0.98
Confusion Matrix:
[[963   2]
 [ 16 134]]
Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       965
        spam       0.99      0.89      0.94       150

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



**Nhận xét:**

### 2.2 Bài tập thực hành 1: Xây dựng mô hình Naïve ngây thơ trên tập dữ liệu hành vi của khách hàng

### 2.3 Bài tập thực hành 2: Xây dựng mô hình Naïve ngây thơ trên tập dữ liệu mushroom.